# Specification curve: communal and individual prayer DIDs

## Researcher degrees of freedom we vary

- sample: full / drop ads with $\le 3$ responses / drop Aïn Sebaâ
- estimator: binomial GLM (logit) / weighted OLS (LPM)
- controls_demog: with / without
- controls_weather: with / without
- location_fe: with / without

That gives $3 \times 2 \times 2 \times 2 \times 2 = 48$ specifications per DID.

We use cluster-robust (CR1) standard errors throughout. The wild cluster bootstrap-$t$ correction lives in `wild_bootstrap.ipynb` and applies only to the preferred specification.

Data/script dependencies: Loads `midsave/df_well_glm.parquet`. Run `did_analysis.ipynb` once before this notebook.

In [ ]:
import itertools
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
REPO = Path('..')

weather_covs = ['temperature_2m','precipitation','cloud_cover','wind_speed_10m','relative_humidity_2m']
demog_covs   = ['is_female','age_25_49','age_50_65']

## Load the analysis dataset

In [ ]:
path = REPO / 'midsave/df_well_glm.parquet'
if not path.exists():
    raise FileNotFoundError(
        f'{path} not found. Run did_analysis.ipynb first.'
    )
df = pd.read_parquet(path).copy()

df['total_responses'] = df['Yes_Well'] + df['No_Well']
df['prop_good']       = df['Yes_Well'] / df['total_responses']
df['after_int']       = df['after'].astype(int)
df['is_muslim']       = (df['Religion'] == 'muslim').astype(int)
df['is_female']       = (df['Gender']   == 'Female').astype(int)
df['age_25_49']       = (df['Age']      == '(25, 49)').astype(int)
df['age_50_65']       = (df['Age']      == '(50, 65)').astype(int)
df['own_prayer_day']  = (
    ((df['Religion']=='muslim')    & (df['day_of_week_start']=='Friday')) |
    ((df['Religion']=='christian') & (df['day_of_week_start']=='Sunday'))
).astype(int)

print(f'ads: {len(df)}, locations: {df.Location.nunique()}, responses: {df.total_responses.sum():,}')

## Helpers: build samples, formulas, and fit one specification

In [ ]:
def apply_sample(data, choice):
    if choice == 'full':
        return data.copy()
    if choice == 'drop_small':
        return data[data['total_responses'] > 3].copy()
    if choice == 'drop_ains':
        return data[data['Location'] != 'Aïn Sebaâ'].copy()
    raise ValueError(choice)

def build_formula(treat_term, controls_d, controls_w, loc_fe, available_data):
    """Assemble RHS string. `treat_term` is e.g. 'after_int * own_prayer_day'."""
    parts = [treat_term]
    if controls_d:
        parts += [c for c in demog_covs if available_data[c].nunique() > 1]
    if controls_w:
        parts += [c for c in weather_covs if available_data[c].nunique() > 1]
    if loc_fe:
        parts.append('C(Location)')
    return ' + '.join(parts)

def fit_one(data, treat_term, var_of_interest, controls_d, controls_w, loc_fe, estimator):
    """Return dict with coef, SE, p, n_ads, G for a single specification."""
    rhs = build_formula(treat_term, controls_d, controls_w, loc_fe, data)
    cluster_kwds = {'groups': data['Location']}
    try:
        if estimator == 'glm_logit':
            m = smf.glm(
                f'Yes_Well + No_Well ~ {rhs}',
                data=data, family=sm.families.Binomial()
            ).fit(cov_type='cluster', cov_kwds=cluster_kwds)
        else:  # wls_lpm
            m = smf.wls(
                f'prop_good ~ {rhs}',
                data=data, weights=data['total_responses'].astype(float)
            ).fit(cov_type='cluster', cov_kwds=cluster_kwds)
        coef = m.params.get(var_of_interest, np.nan)
        se   = m.bse.get(var_of_interest,   np.nan)
        p    = m.pvalues.get(var_of_interest, np.nan)
    except Exception as e:
        coef = se = p = np.nan
    return {
        'coef': coef, 'se': se, 'p': p,
        'n_ads': len(data), 'G': data['Location'].nunique(),
    }

## Run the multiverse

In [ ]:
def run_multiverse(base_data, treat_term, var_of_interest, design_label):
    grid = list(itertools.product(
        ['full','drop_small','drop_ains'],   # sample
        ['glm_logit','wls_lpm'],              # estimator
        [True, False],                        # controls_d
        [True, False],                        # controls_w
        [True, False],                        # loc_fe
    ))
    rows = []
    for sample, estimator, cd, cw, fe in grid:
        sub = apply_sample(base_data, sample)
        res = fit_one(sub, treat_term, var_of_interest, cd, cw, fe, estimator)
        rows.append({
            'design': design_label,
            'sample': sample, 'estimator': estimator,
            'demog_controls': cd, 'weather_controls': cw, 'location_fe': fe,
            **res,
        })
    return pd.DataFrame(rows)

df_comm = df[df['day_of_week_start'].isin(['Friday','Sunday'])].copy()
df_tue  = df[df['day_of_week_start']=='Tuesday'].copy()

mv_comm = run_multiverse(df_comm, 'after_int * own_prayer_day', 'after_int:own_prayer_day',
                          design_label='Communal (Fri+Sun)')
mv_ind  = run_multiverse(df_tue,  'after_int * is_muslim',     'after_int:is_muslim',
                          design_label='Individual (Tue/Dhuhr)')

mv = pd.concat([mv_comm, mv_ind], ignore_index=True)
print('specifications fit:', len(mv))
print('   any NaN coef:', mv['coef'].isna().sum())
mv.head()

## Coefficient summary by design

Note: GLM-logit coefficients are on the logit scale; LPM coefficients are on the probability scale (units of one). They are **not directly comparable in magnitude** but the *sign* and *significance* are.

In [ ]:
for label, sub in mv.groupby('design'):
    print(f'\n=== {label} ===')
    for est, sub2 in sub.groupby('estimator'):
        sig = (sub2['p'] < 0.05).sum()
        pos = (sub2['coef'] > 0).sum()
        print(f'  {est:>10s}: n_specs={len(sub2):>2d}  sig@5%={sig:>2d}  '
              f'coef>0={pos:>2d}  coef range=[{sub2.coef.min():+.4f}, {sub2.coef.max():+.4f}]')

## Specification-curve plot

Top panel: sorted DID coefficients with 95 % CIs and significance markers, separately for each estimator (logit and LPM live on different scales, so we split the curves rather than overlay them). Bottom panel: dot-matrix of the analytic choice corresponding to each specification.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 9, 'axes.titlesize': 11, 'axes.labelsize': 10, 'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 9})
def speccurve(sub, title, ax_top, ax_dot, choice_cols):
    sub = sub.copy().sort_values('coef').reset_index(drop=True)
    x = np.arange(len(sub))
    coef = sub['coef'].values
    ci_lo = coef - 1.96 * sub['se'].values
    ci_hi = coef + 1.96 * sub['se'].values
    sig   = sub['p'].values < 0.05

    # Top: coefficients
    ax_top.errorbar(x[~sig], coef[~sig], yerr=[coef[~sig]-ci_lo[~sig], ci_hi[~sig]-coef[~sig]],
                     fmt='o', color='#DBDBDB', ecolor='#DBDBDB',
                     markersize=3, elinewidth=0.7, label='p≥0.05')
    ax_top.errorbar(x[sig],  coef[sig],  yerr=[coef[sig]-ci_lo[sig], ci_hi[sig]-coef[sig]],
                     fmt='o', color='#E09238', ecolor='#E09238',
                     markersize=4, elinewidth=0.9, label='p<0.05')
    ax_top.axhline(0, color='k', lw=0.7, ls='--')
    ax_top.set_ylabel('DID coefficient')
    ax_top.set_title(title)
    ax_top.legend(loc='upper left', frameon=False)
    ax_top.set_xticks([])
    ax_top.spines[['right','top']].set_visible(False)

    # Bottom: choice dot-matrix
    _short = {'demog_controls':'demog','weather_controls':'weather','location_fe':'loc FE','sample':'sample','estimator':'estimator'}
    rows_dot = []
    for col in choice_cols:
        levels = sorted(sub[col].dropna().astype(str).unique())
        for lvl in levels:
            rows_dot.append((f'{_short.get(col, col)}={lvl}', (sub[col].astype(str) == lvl).values))
    for j, (label, mask) in enumerate(rows_dot):
        xs = x[mask]
        ax_dot.scatter(xs, np.full_like(xs, j, dtype=float),
                       s=10, color='#707070')
    ax_dot.set_yticks(range(len(rows_dot)))
    ax_dot.set_yticklabels([r[0] for r in rows_dot])
    ax_dot.set_xlim(ax_top.get_xlim())
    ax_dot.set_xlabel('Specification (sorted by coefficient)')
    ax_dot.invert_yaxis()
    ax_dot.spines[['right','top']].set_visible(False)

choice_cols = ['sample','estimator','demog_controls','weather_controls','location_fe']

for design_label, mv_sub in mv.groupby('design'):
    for est in ['glm_logit','wls_lpm']:
        sub = mv_sub[mv_sub.estimator == est].copy()
        if sub.empty:
            continue
        fig, (ax_top, ax_dot) = plt.subplots(
            2, 1, figsize=(4.4, 2.9),
            gridspec_kw={'height_ratios':[2.2, 1.4], 'hspace':0.05},
            sharex=True,
        )
        title = f'{design_label} — {est}'
        speccurve(sub, title, ax_top, ax_dot,
                  ['sample','demog_controls','weather_controls','location_fe'])
        fname = f"specurve_{design_label.split()[0].lower()}_{est}.pdf"
        fig.savefig(REPO / 'output' / fname, bbox_inches='tight')
        plt.show()
        # Wider standalone version of the individual GLM-logit panel for the main text (Figure 7)
        if design_label.split()[0].lower() == 'individual' and est == 'glm_logit':
            fig2, (a1, a2) = plt.subplots(
                2, 1, figsize=(6, 4),
                gridspec_kw={'height_ratios':[2.2, 1.4], 'hspace':0.05}, sharex=True,
            )
            speccurve(sub, title, a1, a2,
                      ['sample','demog_controls','weather_controls','location_fe'])
            fig2.savefig(REPO / 'output' / 'specurve_individual_glm_logit_main.pdf', bbox_inches='tight')
            plt.close(fig2)

## Save multiverse table

In [ ]:
out = REPO / 'output/specification_curve.csv'
mv.round(4).to_csv(out, index=False)
print(f'Wrote {out}  ({len(mv)} rows)')

- A specification curve that is **mostly above zero with most points significant** would support an acute prayer effect.
- A curve that is **mostly near zero with few points significant** supports the bounded-null story in the manuscript.
- The dot-matrix below the curve shows which analytic choice produces each end of the distribution. Patterns to watch for:
  - Are all the significant specifications concentrated in a single corner of the choice grid (e.g. only without location FE)? Then the headline finding is fragile to that one decision.
  - Does dropping Aïn Sebaâ flip the sign? The robustness of the bounded null hinges on this.
  - Do GLM-logit and LPM agree on sign across all combinations?